In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import json
import time

# Setup stealth options to avoid 403
options = Options()
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

# Initialize driver
driver = webdriver.Chrome(options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

try:
    # Navigate to URL
    url = 'https://www.partselect.com/PS258119-GE-WD01X10103-Dishwasher-Clamp.htm'
    driver.get(url)
    
    # Wait for page to load
    time.sleep(3)
    
    # Parse with BeautifulSoup for easier extraction
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    
    # Extract product data
    product_data = {}
    
    # Title
    title = soup.find('h1', class_='title-lg')
    product_data['title'] = title.text.strip() if title else None
    
    # Price
    price = soup.find('span', class_='js-partPrice')
    product_data['price'] = price.text.strip() if price else None
    
    # Availability
    availability = soup.find('span', itemprop='availability')
    product_data['availability'] = availability.get('content') if availability else None
    
    # Part Numbers
    ps_number = soup.find('span', class_='bold text-teal', itemprop='productID')
    product_data['partselect_number'] = ps_number.text.strip() if ps_number else None
    
    manufacturer_pn = soup.find('span', class_='bold text-teal', itemprop='mpn')
    product_data['manufacturer_part_number'] = manufacturer_pn.text.strip() if manufacturer_pn else None
    
    # Brand
    brand = soup.find('span', itemprop='brand')
    if brand:
        brand_name = brand.find('span', itemprop='name')
        product_data['brand'] = brand_name.text.strip() if brand_name else None
    
    # Description
    description = soup.find('div', itemprop='description')
    product_data['description'] = description.text.strip() if description else None
    
    # Image URLs
    images = []
    main_img = soup.find('img', itemprop='image')
    if main_img:
        images.append(main_img.get('src'))
    
    # Get all thumbnail images
    thumbs = soup.find_all('a', class_='js-part-img-thumb')
    for thumb in thumbs:
        img_url = thumb.get('data-large-src')
        if img_url and img_url not in images:
            images.append(img_url)
    
    product_data['images'] = images
    
    # Repair Rating
    repair_difficulty = soup.find('div', class_='pd__repair-rating__container__item__icon')
    if repair_difficulty:
        difficulty_text = repair_difficulty.find('p', class_='bold')
        product_data['repair_difficulty'] = difficulty_text.text.strip() if difficulty_text else None
    
    # Repair Duration
    duration = soup.find_all('div', class_='d-flex')
    for div in duration:
        if 'mins' in div.text:
            product_data['repair_duration'] = div.text.strip()
            break
    
    # Replaces
    replaces_section = soup.find('div', class_='bold mb-1', text='Part# WD01X10103 replaces these:')
    if replaces_section:
        replaces_div = replaces_section.find_next_sibling('div')
        product_data['replaces'] = replaces_div.text.strip() if replaces_div else None
    
    # Symptoms
    symptoms = []
    symptoms_section = soup.find('div', class_='bold mb-1', text='This part fixes the following symptoms:')
    if symptoms_section:
        symptom_list = symptoms_section.find_next('ul')
        if symptom_list:
            symptoms = [li.text.strip() for li in symptom_list.find_all('li')]
    product_data['fixes_symptoms'] = symptoms
    
    # Works with products
    product_types = []
    products_section = soup.find('div', class_='bold mb-1', text='This part works with the following products:')
    if products_section:
        product_list = products_section.find_next('ul')
        if product_list:
            product_types = [li.text.strip() for li in product_list.find_all('li')]
    product_data['works_with_products'] = product_types
    
    # Related Parts (You May Also Need)
    related_parts = []
    related_section = soup.find_all('div', class_='pd__related-part')
    for part in related_section:
        part_name = part.find('a', class_='bold')
        part_price = part.find('div', class_='title-md bold mt-2')
        if part_name and part_price:
            related_parts.append({
                'name': part_name.text.strip(),
                'price': part_price.text.strip(),
                'url': 'https://www.partselect.com' + part_name.get('href') if part_name.get('href') else None
            })
    product_data['related_parts'] = related_parts
    
    # Compatible Models (first 10 as example)
    compatible_models = []
    model_rows = soup.find_all('div', class_='row')
    for row in model_rows[:10]:  # Limit to first 10
        cols = row.find_all(['div', 'a'])
        if len(cols) >= 3:
            brand = cols[0].text.strip() if cols[0].name == 'div' else None
            model = cols[1].text.strip() if cols[1].name == 'a' else None
            if brand and model:
                compatible_models.append({
                    'brand': brand,
                    'model': model
                })
    product_data['compatible_models'] = compatible_models
    
    # Metadata
    meta_data = soup.find('div', id='main')
    if meta_data:
        product_data['inventory_id'] = meta_data.get('data-inventory-id')
        product_data['category'] = meta_data.get('data-category')
        product_data['model_type'] = meta_data.get('data-modeltype')
    
    # Print the extracted data
    print(json.dumps(product_data, indent=2))
    
    # Save to file
    with open('product_data.json', 'w') as f:
        json.dump(product_data, f, indent=2)
    
    print("\n✓ Data saved to product_data.json")

finally:
    driver.quit()

C:\Users\trrsh\AppData\Local\Temp\ipykernel_3824\682770306.py:93: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  replaces_section = soup.find('div', class_='bold mb-1', text='Part# WD01X10103 replaces these:')
C:\Users\trrsh\AppData\Local\Temp\ipykernel_3824\682770306.py:100: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  symptoms_section = soup.find('div', class_='bold mb-1', text='This part fixes the following symptoms:')
C:\Users\trrsh\AppData\Local\Temp\ipykernel_3824\682770306.py:109: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  products_section = soup.find('div', class_='bold mb-1', text='This part works with the following products:')


{
  "title": "Dishwasher Clamp WD01X10103",
  "price": "8.92",
  "availability": "InStock",
  "partselect_number": "PS258119",
  "manufacturer_part_number": "WD01X10103",
  "brand": "GE",
  "description": "This clamps the drain pump port to the sump inlet port to hold it in place.",
  "images": [
    "https://partselectcom-gtcdcddbene3cpes.z01.azurefd.net/258119-1-M-GE-WD01X10103-Dishwasher-Clamp.jpg",
    "https://partselectcom-gtcdcddbene3cpes.z01.azurefd.net//258119-1-L-GE-WD01X10103-Dishwasher-Clamp.jpg",
    "https://partselectcom-gtcdcddbene3cpes.z01.azurefd.net//258119-2-L-GE-WD01X10103-Dishwasher-Clamp.jpg",
    "https://partselectcom-gtcdcddbene3cpes.z01.azurefd.net//258119-3-L-GE-WD01X10103-Dishwasher-Clamp.jpg"
  ],
  "repair_difficulty": "Really Easy",
  "repair_duration": "Really Easy\u00a0\n\n\n\n15 - 30 mins",
  "replaces": "AP3423685,  943292,  WD1X10103",
  "fixes_symptoms": [
    "Leaking"
  ],
  "works_with_products": [
    "Dishwasher"
  ],
  "related_parts": [
    

In [10]:
import os
import csv
import json
import time
import random
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

csv_file = "./filtered_urls/products_urls.csv"
output_folder = "./database/product_json_db"
os.makedirs(output_folder, exist_ok=True)

# Selenium options with better stealth
options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# Initialize driver
driver = webdriver.Chrome(options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

# Load product URLs
try:
    with open(csv_file, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        product_urls = [row['url'] for row in reader]
except FileNotFoundError:
    print(f"Error: Could not find {csv_file}")
    driver.quit()
    exit(1)

print(f"Found {len(product_urls)} URLs to scrape")

for idx, url in enumerate(product_urls, 1):
    try:
        print(f"\n[{idx}/{len(product_urls)}] Processing: {url}")
        
        driver.get(url)
        
        # Wait for the main content to load
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CLASS_NAME, "title-lg"))
            )
        except:
            print(f"  ⚠ Timeout waiting for page load")
        
        # Random delay to avoid detection
        time.sleep(random.uniform(2, 4))
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # Extract product data
        product_data = {}
        product_data['url'] = url

        # Title
        title = soup.find('h1', class_='title-lg')
        product_data['title'] = title.text.strip() if title else None

        # Price
        price = soup.find('span', class_='js-partPrice')
        product_data['price'] = price.text.strip() if price else None

        # Availability
        availability = soup.find('span', itemprop='availability')
        product_data['availability'] = availability.get('content') if availability else None

        # Part Numbers
        ps_number = soup.find('span', class_='bold text-teal', itemprop='productID')
        product_data['partselect_number'] = ps_number.text.strip() if ps_number else None

        manufacturer_pn = soup.find('span', class_='bold text-teal', itemprop='mpn')
        product_data['manufacturer_part_number'] = manufacturer_pn.text.strip() if manufacturer_pn else None

        # Brand
        brand = soup.find('span', itemprop='brand')
        if brand:
            brand_name = brand.find('span', itemprop='name')
            product_data['brand'] = brand_name.text.strip() if brand_name else None
        else:
            product_data['brand'] = None

        # Description
        description = soup.find('div', itemprop='description')
        product_data['description'] = description.text.strip() if description else None

        # Image URLs
        images = []
        main_img = soup.find('img', itemprop='image')
        if main_img:
            img_src = main_img.get('src')
            if img_src:
                images.append(img_src)

        # Get all thumbnail images
        thumbs = soup.find_all('a', class_='js-part-img-thumb')
        for thumb in thumbs:
            img_url = thumb.get('data-large-src')
            if img_url and img_url not in images:
                images.append(img_url)

        product_data['images'] = images

        # Repair Rating
        repair_difficulty = soup.find('div', class_='pd__repair-rating__container__item__icon')
        if repair_difficulty:
            difficulty_text = repair_difficulty.find('p', class_='bold')
            product_data['repair_difficulty'] = difficulty_text.text.strip() if difficulty_text else None
        else:
            product_data['repair_difficulty'] = None

        # Repair Duration
        product_data['repair_duration'] = None
        duration_divs = soup.find_all('div', class_='d-flex')
        for div in duration_divs:
            text = div.text
            if 'mins' in text or 'minutes' in text.lower():
                product_data['repair_duration'] = text.strip()
                break

        # Replaces - Fix to search for any part number pattern
        replaces_section = None
        for div in soup.find_all('div', class_='bold mb-1'):
            if 'replaces these:' in div.text:
                replaces_section = div
                break
        
        if replaces_section:
            replaces_div = replaces_section.find_next_sibling('div')
            product_data['replaces'] = replaces_div.text.strip() if replaces_div else None
        else:
            product_data['replaces'] = None

        # Symptoms
        symptoms = []
        symptoms_section = None
        for div in soup.find_all('div', class_='bold mb-1'):
            if 'fixes the following symptoms' in div.text:
                symptoms_section = div
                break
        
        if symptoms_section:
            symptom_list = symptoms_section.find_next('ul')
            if symptom_list:
                symptoms = [li.text.strip() for li in symptom_list.find_all('li')]
        product_data['fixes_symptoms'] = symptoms

        # Works with products
        product_types = []
        products_section = None
        for div in soup.find_all('div', class_='bold mb-1'):
            if 'works with the following products' in div.text:
                products_section = div
                break
        
        if products_section:
            product_list = products_section.find_next('ul')
            if product_list:
                product_types = [li.text.strip() for li in product_list.find_all('li')]
        product_data['works_with_products'] = product_types

        # Related Parts (You May Also Need)
        related_parts = []
        related_section = soup.find_all('div', class_='pd__related-part')
        for part in related_section:
            part_name = part.find('a', class_='bold')
            part_price = part.find('div', class_='title-md bold mt-2')
            if part_name and part_price:
                related_parts.append({
                    'name': part_name.text.strip(),
                    'price': part_price.text.strip().replace('$', '').strip(),
                    'url': 'https://www.partselect.com' + part_name.get('href') if part_name.get('href') else None
                })
        product_data['related_parts'] = related_parts

        # Compatible Models - improved selection
        compatible_models = []
        crossref_section = soup.find('div', class_='pd__crossref__list')
        if crossref_section:
            model_rows = crossref_section.find_all('div', class_='row', recursive=False)
            for row in model_rows[:20]:  # Get first 20
                cols = row.find_all(['div', 'a'], recursive=False)
                if len(cols) >= 2:
                    brand_col = cols[0]
                    model_col = cols[1]
                    
                    brand = brand_col.text.strip() if brand_col else None
                    model = model_col.text.strip() if model_col else None
                    
                    if brand and model:
                        compatible_models.append({
                            'brand': brand,
                            'model': model
                        })
        product_data['compatible_models'] = compatible_models

        # Metadata
        meta_data = soup.find('div', id='main')
        if meta_data:
            product_data['inventory_id'] = meta_data.get('data-inventory-id')
            product_data['category'] = meta_data.get('data-category')
            product_data['model_type'] = meta_data.get('data-modeltype')
        else:
            product_data['inventory_id'] = None
            product_data['category'] = None
            product_data['model_type'] = None

        # Use PS Number or generate filename from URL
        if product_data['partselect_number']:
            filename = product_data['partselect_number']
        elif product_data['manufacturer_part_number']:
            filename = product_data['manufacturer_part_number']
        else:
            # Extract from URL
            filename = url.split('/')[-1].replace('.htm', '')
        
        # Clean filename
        filename = filename.replace("/", "_").replace(" ", "_").replace(":", "_")
        filename = filename + ".json"
        filepath = os.path.join(output_folder, filename)

        # Save to JSON
        with open(filepath, 'w', encoding="utf-8") as f:
            json.dump(product_data, f, indent=2, ensure_ascii=False)

        print(f"  ✓ Saved {filename}")

    except Exception as e:
        print(f"  ✗ Failed for {url}: {str(e)}")
        # Continue to next URL instead of crashing
        continue
    
    # Random delay between requests
    time.sleep(random.uniform(1, 3))

driver.quit()
print("\n✓ All done! Scraped products saved to:", output_folder)

Found 1772 URLs to scrape

[1/1772] Processing: https://www.partselect.com/PS258119-GE-WD01X10103-Dishwasher-Clamp.htm
  ✓ Saved PS258119.json

[2/1772] Processing: https://www.partselect.com/PS259065-GE-WD12X10047-Dishwasher-Float.htm
  ✓ Saved PS259065.json

[3/1772] Processing: https://www.partselect.com/PS259159-GE-WD12X334-Dishwasher-Float.htm
  ✓ Saved PS259159.json

[4/1772] Processing: https://www.partselect.com/PS259256-GE-WD12X448-Dishwasher-Cover-Junction-Box.htm
  ✓ Saved PS259256.json

[5/1772] Processing: https://www.partselect.com/PS259411-GE-WD15X93-Dishwasher-Water-Inlet-Valve.htm
  ✓ Saved PS259411.json

[6/1772] Processing: https://www.partselect.com/PS259679-GE-WD1X1363-Dishwasher-Switch-Leaf.htm
  ✓ Saved PS259679.json

[7/1772] Processing: https://www.partselect.com/PS259749-GE-WD1X1467-Dishwasher-Vent-Pad.htm
  ✓ Saved PS259749.json

[8/1772] Processing: https://www.partselect.com/PS260133-GE-WD21X10078-Dishwasher-Timer-60-Hz.htm
  ✓ Saved PS260133.json

[9/1772]

  ✓ Saved PS468972.json

[64/1772] Processing: https://www.partselect.com/PS468976-Frigidaire-5303917633-Refrigerator-Universal-Defrost-Timer.htm
  ✓ Saved PS468976.json

[65/1772] Processing: https://www.partselect.com/PS468977-Frigidaire-5303917634-Refrigerator-Universal-Defrost-Timer.htm
  ✓ Saved PS468977.json

[66/1772] Processing: https://www.partselect.com/PS469516-Frigidaire-5303918208-Refrigerator-Defrost-Heater-Kit.htm
  ✓ Saved PS469516.json

[67/1772] Processing: https://www.partselect.com/PS471886-Frigidaire-5304413178-Refrigerator-Compressor-Start-Relay.htm
  ✓ Saved PS471886.json

[68/1772] Processing: https://www.partselect.com/PS475011-Frigidaire-5317532406-Refrigerator-Gasket.htm
  ✓ Saved PS475011.json

[69/1772] Processing: https://www.partselect.com/PS475194-Frigidaire-5318680302-Refrigerator-Door-Gasket.htm
  ✓ Saved PS475194.json

[70/1772] Processing: https://www.partselect.com/PS503621-Frigidaire-WF2CB-Refrigerator-Water-Filter.htm
  ✓ Saved PS503621.json

[71/

  ✓ Saved PS2003487.json

[126/1772] Processing: https://www.partselect.com/PS2003553-Whirlpool-12002082-Refrigerator-Flipper-Assembly.htm
  ✓ Saved PS2003553.json

[127/1772] Processing: https://www.partselect.com/PS2003734-Whirlpool-12002311-Refrigerator-Closer.htm
  ✓ Saved PS2003734.json

[128/1772] Processing: https://www.partselect.com/PS2060516-Whirlpool-61005136-Refrigerator-Door-Front-Pick-Off-Shelf.htm
  ✓ Saved PS2060516.json

[129/1772] Processing: https://www.partselect.com/PS2068569-Whirlpool-67004748-Refrigerator-Door-Bucket-Medium.htm
  ✓ Saved PS2068569.json

[130/1772] Processing: https://www.partselect.com/PS2073412-Whirlpool-69168-5-Refrigerator-Door-Handle-Kit.htm
  ✓ Saved PS2073412.json

[131/1772] Processing: https://www.partselect.com/PS2123362-Whirlpool-DA34-10003S-Refrigerator-Compressor-Overload-Protector.htm
  ✓ Saved PS2123362.json

[132/1772] Processing: https://www.partselect.com/PS2180251-Whirlpool-W10175931-Dishwasher-Conversion-Kit.htm
  ✓ Saved PS218


[185/1772] Processing: https://www.partselect.com/PS3407200-Whirlpool-W10254259-Refrigerator-Dispenser-Drip-Tray.htm
  ✓ Saved PS3407200.json

[186/1772] Processing: https://www.partselect.com/PS3408470-Frigidaire-241801801-Refrigerator-Crisper-Pan.htm
  ✓ Saved PS3408470.json

[187/1772] Processing: https://www.partselect.com/PS3408496-Frigidaire-241862405-Refrigerator-Upper-Hinge-Cover.htm
  ✓ Saved PS3408496.json

[188/1772] Processing: https://www.partselect.com/PS3408692-Frigidaire-242068701-Refrigerator-Shelf.htm
  ✓ Saved PS3408692.json

[189/1772] Processing: https://www.partselect.com/PS3412249-Frigidaire-L304432837-Refrigerator-Freezer-Thermometer.htm
  ✓ Saved PS3412249.json

[190/1772] Processing: https://www.partselect.com/PS3412266-Frigidaire-WF3CB-Refrigerator-Water-Filter.htm
  ✓ Saved PS3412266.json

[191/1772] Processing: https://www.partselect.com/PS3417694-GE-WR17X12830-Refrigerator-Shelf.htm
  ✓ Saved PS3417694.json

[192/1772] Processing: https://www.partselect.c

  ✓ Saved PS3529285.json

[246/1772] Processing: https://www.partselect.com/PS3531705-LG-ADD36429802-Door-Foam-Assembly-Refrigerator.htm
  ✓ Saved PS3531705.json

[247/1772] Processing: https://www.partselect.com/PS3532175-LG-AED37082916-Handle-Assembly-Refrigerator.htm
  ✓ Saved PS3532175.json

[248/1772] Processing: https://www.partselect.com/PS3532211-LG-AED72952801-Handle-Assembly-Refrigerator.htm
  ✓ Saved PS3532211.json

[249/1772] Processing: https://www.partselect.com/PS3533582-LG-EAV43060808-Refrigerator-Light-Board-Assembly.htm
  ✓ Saved PS3533582.json

[250/1772] Processing: https://www.partselect.com/PS3534510-LG-MAN61844601-Refrigerator-Door-Basket.htm
  ✓ Saved PS3534510.json

[251/1772] Processing: https://www.partselect.com/PS3534647-LG-MBG38252601-Refrigerator-Dispenser-Lever.htm
  ✓ Saved PS3534647.json

[252/1772] Processing: https://www.partselect.com/PS3535255-LG-MEA32865501-Refrigerator-Ice-Maker-Fill-Tube-Guide.htm
  ✓ Saved PS3535255.json

[253/1772] Processing:

...

  ✓ Saved PS12349161.json

[1530/1772] Processing: https://www.partselect.com/PS12349163-Whirlpool-W11202789-Refrigerator-Auger-Motor.htm
  ✓ Saved PS12349163.json

[1531/1772] Processing: https://www.partselect.com/PS12364147-Frigidaire-241798231-Refrigerator-Ice-Maker-Assembly.htm
  ✓ Saved PS12364147.json

[1532/1772] Processing: https://www.partselect.com/PS12364162-Frigidaire-241987962-Refrigerator-Freezer-Door-Assembly.htm
  ✓ Saved PS12364162.json

[1533/1772] Processing: https://www.partselect.com/PS12364199-Frigidaire-242126602-Refrigerator-Door-Shelf-Bin.htm
  ✓ Saved PS12364199.json

[1534/1772] Processing: https://www.partselect.com/PS12364211-Frigidaire-242218602-Refrigerator-Deli-Drawer-Cover.htm
  ✓ Saved PS12364211.json

[1535/1772] Processing: https://www.partselect.com/PS12364256-Frigidaire-297415203-Refrigerator-Control.htm
  ✓ Saved PS12364256.json

[1536/1772] Processing: https://www.partselect.com/PS12364865-Frigidaire-5304511770-Refrigerator-Handle.htm
  ✓ Saved 

  ✓ Saved PS12722262.json

[1589/1772] Processing: https://www.partselect.com/PS12722272-Samsung-NN34J9902APASH-Refrigerator-Compressor.htm
  ✓ Saved PS12722272.json

[1590/1772] Processing: https://www.partselect.com/PS12722881-GE-WD09X25420-BRONZE-DISHWASHER-HANDLE.htm
  ✓ Saved PS12722881.json

[1591/1772] Processing: https://www.partselect.com/PS12723342-GE-WR02X28072-REFRIGERATOR-FRAME-DOOR-LOWER-MAGNET.htm
  ✓ Saved PS12723342.json

[1592/1772] Processing: https://www.partselect.com/PS12723385-GE-WR12X31693-REFRIGERATOR-DOOR-HANDLE-WHITE.htm
  ✓ Saved PS12723385.json

[1593/1772] Processing: https://www.partselect.com/PS12723433-GE-WR32X31557-REFRIGERATOR-SHELF.htm
  ✓ Saved PS12723433.json

[1594/1772] Processing: https://www.partselect.com/PS12723489-GE-WR78X30557-REFRIGERATOR-DOOR-FRESH-FOOD-RIGHT-HAN.htm
  ✓ Saved PS12723489.json

[1595/1772] Processing: https://www.partselect.com/PS12723959-Whirlpool-W11368721-Refrigerator-Door-Gasket.htm
  ✓ Saved PS12723959.json

[1596/177

  ✓ Saved PS12743966.json

[1648/1772] Processing: https://www.partselect.com/PS12743967-GE-WR13X31786-REFRIGERATOR-DOOR-LEFT-LOWER-HINGE.htm
  ✓ Saved PS12743967.json

[1649/1772] Processing: https://www.partselect.com/PS12743973-GE-WR13X32502-REFRIGERATOR-DOOR-LEFT-UPPER-HINGE.htm
  ✓ Saved PS12743973.json

[1650/1772] Processing: https://www.partselect.com/PS12744060-GE-WR32X31957-Refrigerator-Tuckaway-Shelf-Gray.htm
  ✓ Saved PS12744060.json

[1651/1772] Processing: https://www.partselect.com/PS12744065-GE-WR32X32346-Refrigerator-Crisper-Drawer.htm
  ✓ Saved PS12744065.json

[1652/1772] Processing: https://www.partselect.com/PS12744077-GE-WR51X31996-REFRIGERATOR-DEFROST-HEATER.htm
  ✓ Saved PS12744077.json

[1653/1772] Processing: https://www.partselect.com/PS12744085-GE-WR55X31984-REFRIGERATOR-POWER-SUPPLY-BOARD.htm
  ✓ Saved PS12744085.json

[1654/1772] Processing: https://www.partselect.com/PS12744142-GE-WR60X32071-REFRIGERATOR-EVAPORATOR-FAN.htm
  ✓ Saved PS12744142.json

[1655

  ✓ Saved PS16619583.json

[1707/1772] Processing: https://www.partselect.com/PS16619584-GE-WR71X37788-BOTTOM-REFRIGERATOR-DOOR-BIN.htm
  ✓ Saved PS16619584.json

[1708/1772] Processing: https://www.partselect.com/PS16659920-GE-WR12X40046-BRUSHED-STAINLESS-REFRIGERATOR-HANDLE.htm
  ✓ Saved PS16659920.json

[1709/1772] Processing: https://www.partselect.com/PS16661522-LG-ACQ30341209-Refrigerator-Shelf-Cover.htm
  ✓ Saved PS16661522.json

[1710/1772] Processing: https://www.partselect.com/PS16662122-LG-AHT75335101-SHELF-ASSEMBLY-REFRIGERATOR.htm
  ✓ Saved PS16662122.json

[1711/1772] Processing: https://www.partselect.com/PS16662123-LG-AHT75335102-SHELF-ASSEMBLY-REFRIGERATOR.htm
  ✓ Saved PS16662123.json

[1712/1772] Processing: https://www.partselect.com/PS16662125-LG-AHT75335104-SHELF-ASSEMBLY-REFRIGERATOR.htm
  ✓ Saved PS16662125.json

[1713/1772] Processing: https://www.partselect.com/PS16662126-LG-AHT75335105-SHELF-ASSEMBLY-REFRIGERATOR.htm
  ✓ Saved PS16662126.json

[1714/1772] Pro

  ✓ Saved PS18169683.json

[1764/1772] Processing: https://www.partselect.com/PS18169762-Midea-12531000A00501-Glass-Shelf-Assembly-Of-Refrigerator.htm
  ✓ Saved PS18169762.json

[1765/1772] Processing: https://www.partselect.com/PS18171430-Midea-12831000001016-Refrigerator-Door-Assembly.htm
  ✓ Saved PS18171430.json

[1766/1772] Processing: https://www.partselect.com/PS18171599-Midea-12831000009882-Refrigerator-Left-Door-Assembly.htm
  ✓ Saved PS18171599.json

[1767/1772] Processing: https://www.partselect.com/PS18171617-Midea-12831000011204-Refrigerator-Door-Assembly.htm
  ✓ Saved PS18171617.json

[1768/1772] Processing: https://www.partselect.com/PS18171620-Midea-12831000012741-Refrigerator-Door-Assembly.htm
  ✓ Saved PS18171620.json

[1769/1772] Processing: https://www.partselect.com/PS18172279-Midea-12931000000115-Steel-Wire-Shelf-Of-Refrigerator.htm
  ✓ Saved PS18172279.json

[1770/1772] Processing: https://www.partselect.com/PS18182291-Midea-16331000013621-Refrigerator-Air-Duct-F